# Experimento A1: entorno determinista sin tiempo en el estado

### Carga de librerías

In [1]:
import numpy as np
import random

import entorno
import agentes
import train

### Parámetros del entorno

In [2]:
TAMANOS = {
    'n_estaciones': 2,
    'anclas_por_est': 10,
    'bicis_por_est': 5,    # Estado inicial
    'bicis_por_accion': 2  # Bicis movidas por el agente en c/ accion
}
COSTOS = {
    'accion': 1,
    'accion_invalida': 10,
    'est_vacia': 1,
    'est_llena': 1,
}
TIEMPOS = {
    'viaje': 0,
    'accion': 0,
    'episodio': 100,
    'estado': 1,
    'update': 999
}
PROBS = {
    'origen': [0, 1],
    'destino': np.array(2*[[1, 0]])
}

TRAIN_PARAMS = {
    'alpha_init': 0.1,
    'alpha_step': 1.0,
    'alpha_end':  0.1,

    'eps_init': 0.1,
    'eps_step': 1.0,
    'eps_end':  0.1
}

### Entrenar agentes

In [ ]:
random.seed(10)

env = entorno.Entorno(TAMANOS, COSTOS, TIEMPOS, PROBS)

agente_M = agentes.MonteCarloAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])
_, _, _, _ = train.train(env, agente_M, params=TRAIN_PARAMS, N=10**4, verbose=False)

agente_Q = agentes.QLearningAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])
_, _, _, _ = train.train(env, agente_Q, params=TRAIN_PARAMS, N=10**4, verbose=False)

### Visualizar resultados

In [9]:
print("MONTE CARLO")
print({int(k[1]): np.argmax(v) for k,v in agente_M.q_table.items()})
_, _, _, _ = train.test(env, agente_M.q_table, N=1)
print()
print("Q LEARNING")
print({int(k[1]): np.argmax(v) for k,v in agente_Q.q_table.items()})
_, _, _, _ = train.test(env, agente_Q.q_table, N=1)

MONTE CARLO
{5: 1, 4: 1, 3: 0, 2: 0, 1: 1, 0: 1, 6: 0, 7: 0, 8: 0, 9: 0}
Entrenamiento finalizado en 0.0 minutos.
# Insatisfechos: 0.0 ± 0.0 (0.0 ± 0.0%)
# Prolongados: 0.0 ± 0.0 (0.0 ± 0.0%)
Tiempo desbalanceo: 0.0 ± 0.0 (0.0 ± 0.0%)
Recompensa: -50.0 ± 0.0

Q LEARNING
{5: 0, 4: 0, 3: 0, 2: 0, 1: 1, 0: 1, 6: 0, 7: 1, 8: 0, 9: 0}
Entrenamiento finalizado en 0.0 minutos.
# Insatisfechos: 0.0 ± 0.0 (0.0 ± 0.0%)
# Prolongados: 0.0 ± 0.0 (0.0 ± 0.0%)
Tiempo desbalanceo: 0.0 ± 0.0 (0.0 ± 0.0%)
Recompensa: -48.0 ± 0.0


### Visualizar tabla de valores ($v_{\pi}$) para Q-Learning

In [7]:
v = {int(k[1]): round(np.max(v), 2) for k,v in agente_Q.q_table.items()}
print(dict(sorted(v.items())))

{0: -42.79, 1: -39.2, 2: -42.98, 3: -42.31, 4: -42.23, 5: -42.08, 6: -41.41, 7: -40.73, 8: -39.95, 9: -41.32}


### Entrenar y testear varias veces (robustez ante exploración aleatoria)

In [13]:
random.seed(123)

N = 10**4
n_sims = 10
conteo_M, conteo_Q = 0, 0

for _ in range(n_sims):
    env = entorno.Entorno(TAMANOS, COSTOS, TIEMPOS, PROBS)
    agente_M2 = agentes.MonteCarloAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])
    agente_Q2 = agentes.QLearningAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])

    _, _, _, _ = train.train(env, agente_M2, params=TRAIN_PARAMS, N=N, verbose=False)
    _, m_M, _, _ = train.test(env, agente_M2.q_table, N=1, verbose=False)

    _, _, _, _ = train.train(env, agente_Q2, params=TRAIN_PARAMS, N=N, verbose=False)
    _, m_Q, _, _ = train.test(env, agente_Q2.q_table, N=1, verbose=False)

    if m_M['rewards'][0] == -48:
        conteo_M += 1
    if m_Q['rewards'][0] == -48:
        conteo_Q += 1

print(f"Simulaciones optimas (Monte Carlo): {100*(conteo_M/n_sims):.1f}%")
print(f"Simulaciones optimas (Q-Learning): {100*(conteo_Q/n_sims):.1f}%")

Simulaciones optimas (Monte Carlo): 0.0%
Simulaciones optimas (Q-Learning): 40.0%
